In [ ]:
# 1. Import Libraries
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

In [ ]:
# 2. Download Caltech101 Dataset
dataset_root = "/content/caltech101"

caltech_dataset = datasets.Caltech101(
    root=dataset_root,
    download=True
)

source_root = os.path.join(
    dataset_root, "caltech101", "101_ObjectCategories"
)

print("Dataset Downloaded:", len(caltech_dataset))

In [ ]:
# 3. Split Dataset into Train, Validation and Test
base_dir = "/content/caltech_split"

train_dir = os.path.join(base_dir, "train")
valid_dir = os.path.join(base_dir, "valid")
test_dir = os.path.join(base_dir, "test")

if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

for folder in [train_dir, valid_dir, test_dir]:
    os.makedirs(folder)

random.seed(42)

classes = sorted([
    d for d in os.listdir(source_root)
    if os.path.isdir(os.path.join(source_root, d))
])

for class_name in classes:

    class_path = os.path.join(source_root, class_name)

    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)

    n = len(images)
    train_end = int(0.50 * n)
    valid_end = int(0.75 * n)

    splits = {
        train_dir: images[:train_end],
        valid_dir: images[train_end:valid_end],
        test_dir: images[valid_end:]
    }

    for folder, files in splits.items():

        os.makedirs(os.path.join(folder, class_name))

        for image in files:
            shutil.copy2(
                os.path.join(class_path, image),
                os.path.join(folder, class_name, image)
            )

print("Dataset Split Completed!")

In [ ]:
# 4. Image Preprocessing
image_transforms = {

    "train": transforms.Compose([
        transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),
        transforms.RandomRotation(15),
        transforms.ColorJitter(),
        transforms.RandomHorizontalFlip(),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ]),

    "valid": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ]),

    "test": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])
}

In [ ]:
# 5. Load Datasets
data = {
    "train": datasets.ImageFolder(train_dir, transform=image_transforms["train"]),
    "valid": datasets.ImageFolder(valid_dir, transform=image_transforms["valid"]),
    "test": datasets.ImageFolder(test_dir, transform=image_transforms["test"])
}

batch_size = 32

dataloaders = {
    "train": DataLoader(data["train"], batch_size=batch_size, shuffle=True, num_workers=2),
    "valid": DataLoader(data["valid"], batch_size=batch_size, shuffle=False, num_workers=2),
    "test": DataLoader(data["test"], batch_size=batch_size, shuffle=False, num_workers=2)
}

class_names = data["train"].classes
n_classes = len(class_names)

print("Number of Classes:", n_classes)
print("Training Images:", len(data["train"]))
print("Validation Images:", len(data["valid"]))
print("Testing Images:", len(data["test"]))

In [ ]:
# 6. Display Sample Images
images, labels = next(iter(dataloaders["train"]))

plt.figure(figsize=(10, 6))

for i in range(8):
    image = images[i].permute(1, 2, 0).numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = np.clip(image * std + mean, 0, 1)

    plt.subplot(2, 4, i + 1)
    plt.imshow(image)
    plt.title(class_names[labels[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 7. Load Pretrained VGG16 Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = models.VGG16_Weights.DEFAULT
model = models.vgg16(weights=weights)

for param in model.parameters():
    param.requires_grad = False

n_inputs = model.classifier[6].in_features

model.classifier[6] = nn.Sequential(
    nn.Linear(n_inputs, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, n_classes),
    nn.LogSoftmax(dim=1)
)

model = model.to(device)

print("VGG16 Model Loaded Successfully!")

In [ ]:
# 8. Define Loss and Optimizer
criterion = nn.NLLLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

In [ ]:
# 9. Train and Validate Model
num_epochs = 3

train_losses = []
valid_losses = []
train_accuracies = []
valid_accuracies = []

best_valid_loss = float("inf")
patience = 2
patience_counter = 0

for epoch in range(num_epochs):

    # Training
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, targets in dataloaders["train"]:

        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += targets.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, targets in dataloaders["valid"]:

            images, targets = images.to(device), targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
            total += targets.size(0)

    valid_loss = running_loss / total
    valid_acc = correct / total

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)
    train_accuracies.append(train_acc)
    valid_accuracies.append(valid_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Valid Loss: {valid_loss:.4f} | "
          f"Valid Acc: {valid_acc:.4f}")

    # Early Stopping
    if valid_loss < best_valid_loss:

        best_valid_loss = valid_loss
        patience_counter = 0

        torch.save(model.state_dict(), "/content/best_vgg16_model.pth")

    else:
        patience_counter += 1

        if patience_counter >= patience:
            print("Early Stopping Triggered!")
            break

print("Training Completed!")

In [ ]:
# 10. Load Best Model
model.load_state_dict(
    torch.load("/content/best_vgg16_model.pth", map_location=device)
)

In [ ]:
# 11. Plot Training and Validation Loss
plt.plot(train_losses, label="Training Loss")
plt.plot(valid_losses, label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
# 12. Plot Training and Validation Accuracy
plt.plot(train_accuracies, label="Training Accuracy")
plt.plot(valid_accuracies, label="Validation Accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
# 13. Test Model
model.eval()
correct = 0
total = 0

with torch.no_grad():

    for images, targets in dataloaders["test"]:

        images, targets = images.to(device), targets.to(device)

        outputs = model(images)
        predicted = outputs.argmax(1)

        total += targets.size(0)
        correct += (predicted == targets).sum().item()

test_accuracy = correct / total

print("Test Accuracy:", test_accuracy * 100, "%")

In [ ]:
# 14. Display Predictions
images, targets = next(iter(dataloaders["test"]))

model.eval()

with torch.no_grad():
    outputs = model(images.to(device))
    predictions = outputs.argmax(1).cpu()

plt.figure(figsize=(12, 6))

for i in range(8):

    image = images[i].permute(1, 2, 0).numpy()

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = np.clip(image * std + mean, 0, 1)

    plt.subplot(2, 4, i + 1)
    plt.imshow(image)

    plt.title(
        f"Actual: {class_names[targets[i]]}\n"
        f"Predicted: {class_names[predictions[i]]}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 15. Final Results
print("=" * 40)
print("TRANSFER LEARNING - VGG16")
print("=" * 40)

print("Dataset:", "Caltech101")
print("Model:", "VGG16")
print("Classes:", n_classes)
print("Train Images:", len(data["train"]))
print("Validation Images:", len(data["valid"]))
print("Test Images:", len(data["test"]))
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

print("=" * 40)